### Load Data

In [118]:
import os
import numpy as np

# ✅ Path to your saved .npy file
npy_file_path = "../MediaPipe_landmarks/front_wide_landmarks.npy"

# ✅ Load the NumPy array
landmarks_data = np.load(npy_file_path)

# ✅ Check its shape
print("Shape:", landmarks_data.shape)

# ✅ Inspect first frame
print("First frame landmarks:\n", landmarks_data)


Shape: (303, 33, 3)
First frame landmarks:
 [[[ 0.48366398  0.41511261 -0.02171241]
  [ 0.48546034  0.41167003 -0.01659601]
  [ 0.48655739  0.41169119 -0.01671422]
  ...
  [ 0.47826406  0.64868021  0.02499867]
  [ 0.50740004  0.66243327 -0.01525101]
  [ 0.4616679   0.67000687 -0.02777998]]

 [[ 0.48406368  0.41645023 -0.01893907]
  [ 0.4859589   0.41216728 -0.0122658 ]
  [ 0.48711619  0.41213781 -0.01236048]
  ...
  [ 0.47841749  0.64867872  0.02177846]
  [ 0.50739342  0.65997839 -0.05291202]
  [ 0.46440706  0.65874165 -0.02718856]]

 [[ 0.48429453  0.41700611 -0.03836166]
  [ 0.48608047  0.41239643 -0.03032394]
  [ 0.48739979  0.4123756  -0.03041685]
  ...
  [ 0.47852519  0.64866811  0.00736568]
  [ 0.50740439  0.65866178 -0.06179554]
  [ 0.46546251  0.65633225 -0.0416107 ]]

 ...

 [[ 0.48389536  0.4147349  -0.04418434]
  [ 0.48564738  0.40922958 -0.03795015]
  [ 0.48705718  0.40889385 -0.03802867]
  ...
  [ 0.47881487  0.64757115  0.0191908 ]
  [ 0.50781494  0.65945131 -0.04967593]


### Index different joints and normalize skeleton to be centered by the pelvis and size of entire skeleton 

In [119]:
import sys, os

# Go two levels up to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

print("Project root added to path:", project_root)

from Utils.utils.utils import *



# MediaPipe joint indices
HIP_L = 23
KNEE_L = 25
ANKLE_L = 27
TOE_L = 31
HEEL_L = 29

HIP_R = 24
KNEE_R = 26
ANKLE_R = 28
TOE_R = 32
HEEL_R = 30

SHOULDER_L = 11
ELBOW_L = 13
WRIST_L = 15
THUMB_L = 21
INDEXFINGER_L = 19
PINKY_L = 17

SHOULDER_R = 12
ELBOW_R = 14
WRIST_R = 16
THUMB_R = 22
INDEXFINGER_R = 20
PINKY_R = 18

NOSE = 0



def normalize_skeleton_with_virtual_joints(coords, lhip_idx, rhip_idx,
                                           lsho_idx, rsho_idx, eps=1e-8):
    """
    coords: (T, J, 3) raw 3D landmarks
    lhip_idx, rhip_idx: left/right hip indices
    lsho_idx, rsho_idx: left/right shoulder indices

    Returns:
      coords_norm: (T, J+2, 3) normalized coords including virtual pelvis and neck
      pelvis:      (T, 3) pelvis positions before centering
      neck:        (T, 3) neck positions before centering
    """

    T, J, _ = coords.shape

    # 1) Virtual mid-pelvis and mid-shoulder (neck proxy)
    left_hip  = coords[:, lhip_idx, :]    # (T, 3)
    right_hip = coords[:, rhip_idx, :]    # (T, 3)
    pelvis = (left_hip + right_hip) / 2.0 # (T, 3)

    left_sho  = coords[:, lsho_idx, :]    # (T, 3)
    right_sho = coords[:, rsho_idx, :]    # (T, 3)
    neck = (left_sho + right_sho) / 2.0   # (T, 3)

    # 2) Center all original joints on pelvis
    coords_centered = coords - pelvis[:, None, :]  # (T, J, 3)

    # 3) Also center virtual joints
    pelvis_centered = pelvis - pelvis              # becomes (T, 3) at origin
    neck_centered   = neck - pelvis                # neck relative to pelvis

    # 4) Stack virtual joints at the end: [J joints, pelvis, neck]
    pelvis_centered = pelvis_centered[:, None, :]  # (T, 1, 3)
    neck_centered   = neck_centered[:, None, :]    # (T, 1, 3)
    coords_with_virtual = np.concatenate(
        [coords_centered, pelvis_centered, neck_centered], axis=1
    )  # (T, J+2, 3)

    # 5) Compute scale as pelvis->neck distance
    # neck is last joint index: J+1
    neck_rel = coords_with_virtual[:, -1, :]                # (T, 3)
    scale = np.linalg.norm(neck_rel, axis=-1, keepdims=True) + eps  # (T, 1)

    # 6) Scale all joints
    coords_norm = coords_with_virtual / scale[:, None, :]   # (T, J+2, 3)

    return coords_norm, pelvis, neck



coords_norm, pelvis_raw, neck_raw = normalize_skeleton_with_virtual_joints(
    landmarks_data, HIP_L, HIP_R, SHOULDER_L, SHOULDER_R
)

Project root added to path: c:\Users\chris\OneDrive\Desktop\Fritidsprojekt\TempAISpotter\AI\OlympicAi


### Transform landmark coordinates into angles of different joints. We use angles for our embedding

### Do i need to change to angles from anatomical to flexion?

In [120]:
def compute_angle_features(landmarks):
    frames, joints, dims = landmarks.shape
    features = []

    for f in range(frames):
        lm = landmarks[f]

        # Example angles
        left_ankle = calculate_angle(lm[KNEE_L], lm[ANKLE_L], lm[TOE_L])
        right_ankle = calculate_angle(lm[KNEE_R], lm[ANKLE_R], lm[TOE_R])
        
        left_knee = calculate_angle(lm[HIP_L], lm[KNEE_L], lm[ANKLE_L])
        right_knee = calculate_angle(lm[HIP_R], lm[KNEE_R], lm[ANKLE_R])
        
        left_hip = calculate_angle(lm[SHOULDER_L], lm[HIP_L], lm[KNEE_L])
        right_hip = calculate_angle(lm[SHOULDER_R], lm[HIP_R], lm[KNEE_R])
        
        left_shoulder = calculate_angle(lm[ELBOW_L], lm[SHOULDER_L], lm[HIP_L])
        right_shoulder = calculate_angle(lm[ELBOW_R], lm[SHOULDER_R], lm[HIP_R])

        left_elbow = calculate_angle(lm[SHOULDER_L], lm[ELBOW_L], lm[WRIST_L])
        right_elbow = calculate_angle(lm[SHOULDER_R], lm[ELBOW_R], lm[WRIST_R])
        
        left_wrist = calculate_angle(lm[ELBOW_L], lm[WRIST_L], lm[PINKY_L])
        right_wrist = calculate_angle(lm[ELBOW_R], lm[WRIST_R], lm[PINKY_R])

        # Add more angles if you want a richer embedding

        features.append([
            left_ankle,
            right_ankle,
            left_knee,
            right_knee,
            left_hip,
            right_hip,
            left_shoulder,
            right_shoulder,
            left_wrist,
            right_wrist,
            right_elbow,
            left_elbow,
        ])

    return np.array(features)

new_arr = compute_angle_features(landmarks_data)
new_arr.shape

(303, 12)

In [121]:
new_arr

array([[179.99581915, 157.91728148, 171.50709892, ..., 171.76324723,
         57.02751946,  22.14453229],
       [178.51022066, 153.99815365, 173.60473793, ..., 176.65961172,
         41.00477706,  24.80022497],
       [177.95462505, 151.83103446, 175.00606675, ..., 174.37270752,
         33.97733724,  25.2634832 ],
       ...,
       [176.74261002, 157.57293181, 179.99020685, ..., 170.86317743,
         23.46351353,  27.86515786],
       [176.5868111 , 157.75517067, 179.84850372, ..., 171.53165221,
         23.59876376,  27.96934006],
       [176.39289563, 157.58670039, 179.59442188, ..., 172.80769858,
         22.98853525,  28.70592569]])

In [122]:
new_arr.shape

(303, 12)

### Normalize values from pure angles to (something that i need to check what it gets turned into)

In [123]:
#norm_arr = (new_arr - new_arr.mean(axis=0)) / new_arr.std(axis=0)
#norm_arr

### Add a feature which tells the velocity of movement. Used to compare speed of reps

In [124]:
velocity = np.diff(new_arr, axis=0)
velocity

array([[-1.48559849e+00, -3.91912782e+00,  2.09763901e+00, ...,
         4.89636448e+00, -1.60227424e+01,  2.65569268e+00],
       [-5.55595612e-01, -2.16711920e+00,  1.40132882e+00, ...,
        -2.28690419e+00, -7.02743982e+00,  4.63258236e-01],
       [-1.79701458e+00, -1.15601986e-01, -5.16502863e-02, ...,
        -5.14156273e+00, -7.61914106e+00,  5.55696310e+00],
       ...,
       [ 9.68510030e-03,  3.10136744e-01,  1.63879570e-02, ...,
         4.11184775e-01,  2.35104252e-01,  5.27217532e-03],
       [-1.55798924e-01,  1.82238867e-01, -1.41703128e-01, ...,
         6.68474775e-01,  1.35250237e-01,  1.04182199e-01],
       [-1.93915469e-01, -1.68470281e-01, -2.54081840e-01, ...,
         1.27604638e+00, -6.10228518e-01,  7.36585622e-01]])

### Add feature of knee and hip symmetry

In [125]:
left_knee  = new_arr[:, 3]
right_knee = new_arr[:, 4]
left_hip   = new_arr[:, 5]
right_hip  = new_arr[:, 6]



knee_symmetry = left_knee - right_knee
hip_symmetry  = left_hip - right_hip
knee_symmetry.shape, hip_symmetry.shape

((303,), (303,))

### Smoothed the angles to create less noise in the data. (Maybe this is only relevant if we feed it through a ML pipeline?)

In [126]:
from scipy.signal import savgol_filter
angles_smooth = savgol_filter(new_arr, window_length=11, polyorder=3, axis=0)
angles_smooth

array([[180.18965716, 156.97554358, 171.77824108, ..., 173.29010284,
         57.2562129 ,  22.23987167],
       [178.51459434, 154.65194746, 173.48099638, ..., 173.32840209,
         41.76041874,  24.31309979],
       [177.27202257, 153.1843362 , 174.47420196, ..., 173.71594421,
         31.376525  ,  26.47042753],
       ...,
       [176.67445386, 157.47786206, 179.93229837, ..., 170.99513598,
         23.37513381,  27.92215019],
       [176.57380262, 157.61044193, 179.82180355, ..., 171.6632932 ,
         23.31971949,  28.16635719],
       [176.42427046, 157.69543221, 179.62904019, ..., 172.65537671,
         23.16804496,  28.57165072]])

### Make a new embedding with the new features we created

In [127]:
min_frames = velocity.shape[0]  # 415

# Remove last frames to make vectors match in dimensions
angles_smooth_trimmed = angles_smooth[:min_frames]
knee_symmetry_trimmed = knee_symmetry[:min_frames]
hip_symmetry_trimmed  = hip_symmetry[:min_frames]

# Add dimension for concatenation
knee_symmetry_trimmed = knee_symmetry_trimmed.reshape(-1, 1)
hip_symmetry_trimmed  = hip_symmetry_trimmed.reshape(-1, 1)


velocity.shape, angles_smooth_trimmed.shape, knee_symmetry_trimmed.shape, hip_symmetry_trimmed.shape

embedding = np.concatenate([
    angles_smooth_trimmed,
    velocity,                  # already 415
    knee_symmetry_trimmed,
    hip_symmetry_trimmed
], axis=1)

embedding.shape

(302, 26)

In [128]:
np.save("../embedding/front_wide_embedding.npy", embedding)

In [129]:
bob = np.load("../embedding/front_narrow_embedding.npy")
bobby = np.load("../embedding/front_wide_embedding.npy")

In [130]:
bob.shape, bobby.shape

((413, 26), (302, 26))

In [131]:
def normalize(x):
    return (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-8)

bob = normalize(bob)
bobby = normalize(bobby)


### Use DTW to compare similarity of videos (figure out if i should use cosine, euclidean or manhatten distance metric)

In [142]:
from dtw import dtw
from scipy.spatial.distance import cosine, euclidean
dist, cost, acc, path = dtw(bob, bobby, dist=lambda x, y: cosine(x, y))  # Cosine similarity per frame
dist, path

(109.40190744920763,
 (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
          13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
          26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
          39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
          52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
          65,  66,  67,  68,  69,  70,  70,  71,  72,  73,  74,  75,  75,
          76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,
          89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101,
         101, 102, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
         113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
         139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151,
         152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164,
         165, 166

In [143]:
dist_norm = dist / len(path[0])
similarity = np.exp(-dist_norm)
similarity

0.7780832105596106

### Map which frames correspond to which frame between the two videos

In [134]:
print(path[0])  # Indices in bob
print(path[1])  # Indices in bobby

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

### Calculate per-feature differences. Used to create data which can be used to find the specific deviations between videos (eg not deep enough squat based on not low enough knee angle)

In [135]:
idx_a, idx_b = path

aligned_diffs = []

for i, j in zip(idx_a, idx_b):
    diff = bob[i] - bobby[j]   # signed difference
    aligned_diffs.append(diff)

aligned_diffs = np.array(aligned_diffs)
# shape: (path_length, D)
aligned_diffs

array([[-0.32859602, -0.03418217,  0.2537247 , ...,  0.266556  ,
        -0.86419195,  0.07688996],
       [-0.01382986,  0.40075619,  0.17111139, ...,  0.43275515,
        -0.52959657, -0.23304744],
       [ 0.2223662 ,  0.66238377,  0.12284499, ...,  0.54867228,
        -0.11233581, -0.21982539],
       ...,
       [-0.25847479, -0.41321669, -0.13925706, ..., -0.01794685,
        -0.17125114, -0.06823809],
       [-0.25417195, -0.44219663, -0.13877621, ..., -0.02244405,
        -0.16886868, -0.06264206],
       [-0.23761347, -0.46496175, -0.13431111, ...,  0.06885873,
        -0.15992751, -0.04675784]])

### Find which feature contribute most to the error

In [136]:
feature_error = np.mean(np.abs(aligned_diffs), axis=0)

feature_error

array([0.34437126, 0.28224467, 0.1295364 , 0.14538902, 0.10892612,
       0.13245273, 0.46352728, 0.29874265, 0.94883622, 0.63158459,
       0.30788512, 0.31148368, 0.61794112, 0.61082102, 0.18003809,
       0.26589643, 0.16538105, 0.28861766, 0.65525263, 0.47031225,
       0.62698185, 0.743197  , 0.4826245 , 0.55024355, 0.1711147 ,
       0.15314184])

### Find out when the difference occurs. Can be used to plot the difference in time.

In [137]:
knee_diff_over_time = np.abs(
    aligned_diffs[:, [0, 1]]
).mean(axis=1)


In [138]:
peak_idx = np.argmax(knee_diff_over_time)
peak_idx

153

In [139]:
bob_frame = idx_a[peak_idx]
bobby_frame = idx_b[peak_idx]
bob_frame, bobby_frame

(153, 94)

In [140]:
bob[:, 2]

array([ 4.63672624e-01,  4.63101755e-01,  4.62690157e-01,  4.62345378e-01,
        4.61974968e-01,  4.61486477e-01,  4.60527985e-01,  4.59365005e-01,
        4.57770118e-01,  4.56625064e-01,  4.55673194e-01,  4.54993757e-01,
        4.54252428e-01,  4.53534581e-01,  4.52678618e-01,  4.51646569e-01,
        4.50502360e-01,  4.49679585e-01,  4.49132799e-01,  4.49168431e-01,
        4.50324410e-01,  4.52636250e-01,  4.55653028e-01,  4.59064533e-01,
        4.62622347e-01,  4.65993313e-01,  4.68934437e-01,  4.71133443e-01,
        4.72346820e-01,  4.72963829e-01,  4.73368695e-01,  4.73799616e-01,
        4.74343499e-01,  4.75219096e-01,  4.76456890e-01,  4.81301143e-01,
        4.85368995e-01,  4.87650660e-01,  4.85354898e-01,  4.73188782e-01,
        4.52368098e-01,  4.25023265e-01,  3.86906680e-01,  3.38611327e-01,
        2.89635464e-01,  2.46029468e-01,  2.05625208e-01,  1.71491364e-01,
        1.40326985e-01,  1.07651340e-01,  7.27237523e-02,  3.61528372e-02,
       -1.25334685e-02, -